# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MahboobAli1/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# Two Paper Findings + My Methodology Questions

## Finding 1

The FlyRank research highlights that content freshness is an important factor when identifying pages that may benefit from updates.

### Methodology Question

The prediction target in this project is the Week 4 rule-based baseline score, which combines content age, days since last update, and traffic trend. This makes the label transparent and reproducible.

### Validation Question

A random train/test split provides an initial estimate of model performance, while grouped validation offers a more realistic assessment of how well the model generalizes across different clients.

---

## Finding 2

Search visibility and engagement metrics provide useful signals for prioritizing refresh opportunities.

### Methodology Question

Features such as impressions, clicks, CTR, average position, and engagement rate were included because they represent observable search performance.

### Validation Question

The results identify statistical relationships within the available dataset and should be interpreted as decision-support rather than evidence of causal effects.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [8]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)
# Recreate the Week 4 baseline score
df["content_age_days"] = df["content_age_days"].fillna(0)
df["days_since_last_update"] = df["days_since_last_update"].fillna(0)
df["trend_pct"] = df["trend_pct"].fillna(0)

df["age_score"] = df["content_age_days"] / df["content_age_days"].max()
df["update_score"] = df["days_since_last_update"] / df["days_since_last_update"].max()
df["decline_score"] = (
    np.maximum(-df["trend_pct"], 0) /
    np.maximum(-df["trend_pct"], 0).max()
)

df["baseline_score"] = (
    0.40 * df["age_score"] +
    0.35 * df["update_score"] +
    0.25 * df["decline_score"]
)

features = [
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate"
]

X = df[features].fillna(0)
y = df["baseline_score"]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Grouped Split Results")
print("---------------------")
print("MAE :", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
print("R²  :", r2_score(y_test, pred))

Grouped Split Results
---------------------
MAE : 0.07336402071802976
RMSE: 0.08881335181023421
R²  : 0.5578869201882073


# Honest Validation Summary

The Week 5 model was originally evaluated using a random train/test split.

For this validation audit, the same model was evaluated using a grouped split based on `client_id`, ensuring that pages from the same client were not shared between the training and testing sets.

## Comparison

| Validation Strategy | MAE | RMSE | R² |
|---------------------|------|------|------|
| Random Split | 0.0569 | 0.0726 | 0.6447 |
| Grouped Split | 0.0734 | 0.0888 | 0.5579 |

The grouped split produced slightly lower performance, which is expected because it presents a more challenging and realistic evaluation scenario. This suggests that the Week 5 random split may have produced slightly optimistic estimates.

For this reason, grouped validation provides stronger evidence that the model can generalize to unseen clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

# Leakage Audit

The final feature set was reviewed to identify potential sources of target leakage.

## Excluded Columns

- content_id
- client_id

These columns uniquely identify records or clients and do not provide useful predictive information.

## Future Information

No future observations or unavailable production data were used during training.

## Target Leakage

The prediction target (`baseline_score`) was excluded from the feature set.

## Conclusion

No obvious target leakage was identified in the final model, making the evaluation suitable for decision-support purposes.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [11]:
print("Training Features:")
for feature in features:
    print("-", feature)

Training Features:
- content_age_days
- days_since_last_update
- search_volume
- impressions_90d
- clicks_90d
- sessions_90d
- ctr
- avg_position
- engagement_rate


# Claim Rewrite

## Original Claim

The model accurately identifies which pages should be refreshed.

## Revised Claim

The model estimates content refresh priority by learning patterns present in historical search and engagement data.

The recommendations are intended to support editorial decision-making rather than replace human judgment.

The reported performance reflects the selected validation strategy and should not be interpreted as evidence of Google's ranking algorithm or causal relationships.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.